<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/reinforcement_learning/researcher_agent_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Researcher Agent with LangGraph

This notebook demonstrates building an intelligent **Researcher Agent** using LangGraph and LangChain that can:

- Accept natural language research queries
- Search the web for relevant information
- Synthesize findings into comprehensive research reports

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    RESEARCHER AGENT                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   [Query] ──► [Planner] ──► [Web Search] ──► [Synthesizer]  │
│                   │              │               │          │
│                   ▼              ▼               ▼          │
│              Research        Search          Final          │
│               Plan          Results         Report          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```


## 1. Environment Setup

First, let's install the required dependencies.


In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langchain-anthropic langchain-google-genai langgraph tavily-python


In [ ]:
import os
from typing import Annotated, List, TypedDict, Literal
from datetime import datetime

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

# LangGraph imports
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# Web search
from tavily import TavilyClient

print("✅ All imports successful!")


## 2. API Key Configuration

Set up your API keys. You can use **OpenAI**, **Anthropic**, or **Google Gemini** as the LLM backend.

For web search, we use **Tavily** (get a free API key at https://tavily.com)


In [ ]:
# Option 1: Set API keys directly (not recommended for production)
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key"
# os.environ["GOOGLE_API_KEY"] = "your-google-api-key"
# os.environ["TAVILY_API_KEY"] = "your-tavily-api-key"

# Option 2: Use getpass for secure input
from getpass import getpass

# Choose your LLM provider
LLM_PROVIDER = "openai"  # Options: "openai", "anthropic", "google"

if LLM_PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
elif LLM_PROVIDER == "anthropic" and "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
elif LLM_PROVIDER == "google" and "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

print(f"✅ Configured for {LLM_PROVIDER.upper()} with Tavily web search")


## 3. Initialize the LLM

We support multiple LLM providers. Select one based on your preference.


In [ ]:
def get_llm(provider: str = "openai"):
    """Initialize the LLM based on the selected provider."""

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model="gpt-4o",
            temperature=0.7,
        )

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(
            model="claude-sonnet-4-20250514",
            temperature=0.7,
        )

    elif provider == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(
            model="gemini-1.5-pro",
            temperature=0.7,
        )

    else:
        raise ValueError(f"Unknown provider: {provider}")

# Initialize the LLM
llm = get_llm(LLM_PROVIDER)
print(f"✅ LLM initialized: {llm.model_name if hasattr(llm, 'model_name') else LLM_PROVIDER}")


## 4. Define Web Search Tools

We'll create a powerful web search tool using the Tavily API.


In [ ]:
# Initialize Tavily client
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def web_search(query: str) -> str:
    """Search the web for current information on a given topic.

    Args:
        query: The search query to look up on the web.

    Returns:
        A summary of search results with relevant information.
    """
    try:
        # Perform the search
        response = tavily_client.search(
            query=query,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
            include_raw_content=False,
        )

        # Format the results
        results = []

        # Include the AI-generated answer if available
        if response.get("answer"):
            results.append(f"**Quick Answer:** {response['answer']}\n")

        # Include individual search results
        results.append("**Search Results:**\n")
        for i, result in enumerate(response.get("results", []), 1):
            title = result.get("title", "No title")
            url = result.get("url", "")
            content = result.get("content", "No content available")
            results.append(f"{i}. **{title}**\n   URL: {url}\n   {content}\n")

        return "\n".join(results)

    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def get_current_date() -> str:
    """Get the current date and time for context in research.

    Returns:
        The current date and time as a formatted string.
    """
    return datetime.now().strftime("%B %d, %Y at %H:%M")


# List of tools available to the agent
tools = [web_search, get_current_date]

print("✅ Tools defined:")
for t in tools:
    print(f"   - {t.name}: {t.description[:60]}...")


## 5. Define Agent State

LangGraph uses a typed state object to track the agent's progress through the workflow.


In [ ]:
class ResearcherState(TypedDict):
    """State object for the Researcher Agent.

    Attributes:
        messages: The conversation history with tool calls and responses
        research_query: The original research query from the user
        research_plan: The generated plan for conducting research
        search_results: Accumulated search results
        final_report: The synthesized research report
        iteration_count: Number of research iterations performed
    """
    messages: Annotated[list, add_messages]
    research_query: str
    research_plan: str
    search_results: List[str]
    final_report: str
    iteration_count: int

print("✅ State schema defined")


## 6. Define Agent Nodes

Each node represents a distinct step in the research workflow.


In [ ]:
# System prompts for different agent roles

PLANNER_PROMPT = """You are a research planning expert. Your job is to analyze a research query
and create a structured plan for investigating the topic.

Given the user's research query, create a clear plan that includes:
1. Key aspects of the topic to investigate
2. Specific search queries to execute (2-4 queries)
3. What information to look for in each search

Keep the plan focused and actionable. Format it clearly with numbered steps.

Research Query: {query}
"""

RESEARCHER_PROMPT = """You are an expert researcher with access to web search tools.
Your goal is to find accurate, up-to-date information on the given topic.

Research Plan:
{plan}

Use the web_search tool to find relevant information. Search systematically according
to the research plan. Be thorough but focused.

When you have gathered enough information (typically 2-3 searches), indicate that
you're ready to synthesize by saying "RESEARCH COMPLETE".
"""

SYNTHESIZER_PROMPT = """You are an expert at synthesizing research findings into clear,
comprehensive reports.

Original Query: {query}

Research Findings:
{findings}

Create a well-structured research report that:
1. Provides a clear executive summary
2. Organizes findings into logical sections
3. Highlights key insights and facts
4. Notes any limitations or areas needing further research
5. Includes relevant sources where applicable

Format the report professionally with headers and bullet points.
"""

print("✅ Prompts defined")


In [ ]:
def plan_research(state: ResearcherState) -> ResearcherState:
    """Create a research plan based on the query."""
    print("📋 Planning research...")

    query = state["research_query"]

    # Generate research plan
    plan_prompt = ChatPromptTemplate.from_template(PLANNER_PROMPT)
    chain = plan_prompt | llm | StrOutputParser()
    plan = chain.invoke({"query": query})

    print(f"\n{plan}\n")

    return {
        **state,
        "research_plan": plan,
        "messages": [
            SystemMessage(content=RESEARCHER_PROMPT.format(plan=plan)),
            HumanMessage(content=f"Please research: {query}")
        ],
    }


def conduct_research(state: ResearcherState) -> ResearcherState:
    """Agent node that decides whether to search or synthesize."""
    print(f"🔍 Researching (iteration {state['iteration_count'] + 1})...")

    # Bind tools to the LLM
    llm_with_tools = llm.bind_tools(tools)

    # Get the agent's response
    response = llm_with_tools.invoke(state["messages"])

    return {
        **state,
        "messages": [response],
        "iteration_count": state["iteration_count"] + 1,
    }


def synthesize_report(state: ResearcherState) -> ResearcherState:
    """Synthesize all research findings into a final report."""
    print("📝 Synthesizing final report...")

    # Extract all search results from messages
    findings = []
    for msg in state["messages"]:
        if hasattr(msg, 'content') and msg.content:
            # Look for tool results (search results)
            if isinstance(msg.content, str) and "Search Results" in msg.content:
                findings.append(msg.content)
            elif hasattr(msg, 'tool_calls') or "**" in str(msg.content):
                findings.append(str(msg.content))

    # Combine all findings
    all_findings = "\n\n---\n\n".join(findings) if findings else "No specific findings collected."

    # Generate the final report
    synth_prompt = ChatPromptTemplate.from_template(SYNTHESIZER_PROMPT)
    chain = synth_prompt | llm | StrOutputParser()

    report = chain.invoke({
        "query": state["research_query"],
        "findings": all_findings
    })

    return {
        **state,
        "final_report": report,
        "messages": [AIMessage(content=report)],
    }


print("✅ Agent nodes defined")


## 7. Define Routing Logic

The router determines the next step based on the current state.


In [ ]:
def should_continue_research(state: ResearcherState) -> Literal["tools", "synthesize"]:
    """Determine whether to continue researching or synthesize results."""

    messages = state["messages"]
    last_message = messages[-1]

    # Check if we've hit the iteration limit
    if state["iteration_count"] >= 5:
        print("⏹️ Max iterations reached, moving to synthesis...")
        return "synthesize"

    # Check if the agent wants to use tools
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print(f"🔧 Agent requesting tool: {last_message.tool_calls[0]['name']}")
        return "tools"

    # Check if research is complete
    if hasattr(last_message, 'content') and "RESEARCH COMPLETE" in str(last_message.content).upper():
        print("✅ Research complete, moving to synthesis...")
        return "synthesize"

    # Default: continue to synthesis if no tools needed
    return "synthesize"


print("✅ Routing logic defined")


## 8. Build the LangGraph Workflow

Now we assemble all components into a complete graph.


In [ ]:
def build_researcher_agent():
    """Construct the Researcher Agent graph."""

    # Create the graph
    workflow = StateGraph(ResearcherState)

    # Add nodes
    workflow.add_node("planner", plan_research)
    workflow.add_node("researcher", conduct_research)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_node("synthesizer", synthesize_report)

    # Define the workflow edges
    workflow.add_edge(START, "planner")
    workflow.add_edge("planner", "researcher")

    # Conditional routing from researcher
    workflow.add_conditional_edges(
        "researcher",
        should_continue_research,
        {
            "tools": "tools",
            "synthesize": "synthesizer",
        }
    )

    # After tools, go back to researcher
    workflow.add_edge("tools", "researcher")

    # Synthesizer ends the workflow
    workflow.add_edge("synthesizer", END)

    # Compile the graph
    return workflow.compile()


# Build the agent
researcher_agent = build_researcher_agent()

print("✅ Researcher Agent built successfully!")


In [ ]:
# Visualize the graph (optional - requires graphviz)
try:
    from IPython.display import Image, display
    display(Image(researcher_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization not available. Install graphviz for visualization.")
    print("\nWorkflow structure:")
    print("START → planner → researcher ⟷ tools → synthesizer → END")


## 9. Run the Researcher Agent

Let's test our agent with a research query!


In [ ]:
def run_research(query: str) -> str:
    """Execute a research query and return the final report.

    Args:
        query: The research question or topic to investigate.

    Returns:
        The final research report as a string.
    """
    print("="*60)
    print(f"🔬 RESEARCH AGENT")
    print("="*60)
    print(f"\n📌 Query: {query}\n")
    print("-"*60)

    # Initialize state
    initial_state = {
        "messages": [],
        "research_query": query,
        "research_plan": "",
        "search_results": [],
        "final_report": "",
        "iteration_count": 0,
    }

    # Run the agent
    final_state = researcher_agent.invoke(initial_state)

    print("\n" + "="*60)
    print("📄 FINAL RESEARCH REPORT")
    print("="*60)
    print(final_state["final_report"])

    return final_state["final_report"]


In [ ]:
# Example Research Query 1: Technology
report = run_research(
    "What are the latest developments in AI agents and autonomous systems in 2025? "
    "Focus on practical applications and key players in the industry."
)


In [ ]:
# Example Research Query 2: Science
report = run_research(
    "What are the most promising breakthroughs in renewable energy technology? "
    "Include information about solar, wind, and battery storage innovations."
)


In [ ]:
# Try your own research query!
custom_query = "What are the current trends in machine learning for healthcare applications?"

report = run_research(custom_query)


## 10. Advanced: Streaming Output

For better UX, we can stream the agent's progress in real-time.


In [ ]:
def run_research_streaming(query: str):
    """Execute a research query with streaming output."""
    print("="*60)
    print(f"🔬 RESEARCH AGENT (Streaming Mode)")
    print("="*60)
    print(f"\n📌 Query: {query}\n")
    print("-"*60)

    initial_state = {
        "messages": [],
        "research_query": query,
        "research_plan": "",
        "search_results": [],
        "final_report": "",
        "iteration_count": 0,
    }

    # Stream through the graph
    for event in researcher_agent.stream(initial_state):
        for node_name, node_output in event.items():
            if node_name == "synthesizer" and "final_report" in node_output:
                print("\n" + "="*60)
                print("📄 FINAL RESEARCH REPORT")
                print("="*60)
                print(node_output["final_report"])


# Run with streaming
run_research_streaming("What are the key features and capabilities of LangGraph for building AI agents?")


## 11. Summary

In this notebook, we built a **Researcher Agent** using LangGraph with the following components:

### Architecture
- **Planner Node**: Analyzes the query and creates a research plan
- **Researcher Node**: Executes searches based on the plan
- **Tools Node**: Handles web search tool execution
- **Synthesizer Node**: Compiles findings into a comprehensive report

### Key Features
- ✅ Multi-provider LLM support (OpenAI, Anthropic, Google)
- ✅ Real-time web search via Tavily API
- ✅ Structured state management with TypedDict
- ✅ Conditional routing for iterative research
- ✅ Professional report synthesis
- ✅ Streaming output support

### Next Steps
- Add more tools (document analysis, academic search)
- Implement memory for multi-session research
- Add citation tracking and verification
- Create a web interface with Gradio or Streamlit


In [ ]:
print("\n🎉 Researcher Agent Example Complete!")
print("\nFeel free to modify the prompts, add more tools, or integrate with other APIs!")
